# Model Comparison: VAE Variants vs Heston — AAPL

Compare all 7 models against market IV surfaces for **AAPL**:
- **MLP** / **MLP-log** / **MLP-log-arb** — MLP VAE variants
- **Conv** / **Conv-log** / **Conv-log-arb** — Conv VAE variants
- **Heston** — Calibrated Heston model

**Prerequisites:** Run `python scripts/run_pipeline.py --ticker AAPL --stages eval heston compare`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from collections import OrderedDict

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

# ═══════════════════════════════════════════════
#  CONFIGURATION — change TICKER to reuse
# ═══════════════════════════════════════════════
TICKER = "AAPL"

ROOT = Path("../..") 
EVAL_DIR = ROOT / "artifacts" / "eval" / TICKER
HESTON_DIR = ROOT / "data" / "processed" / "heston" / "surfaces"
OUTPUT_DIR = ROOT / "artifacts" / "comparison" / TICKER
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# VAE model directories
VAE_DIRS = OrderedDict([
    ("MLP",          EVAL_DIR / "mlp" / "surfaces"),
    ("MLP-log",      EVAL_DIR / "mlp_log" / "surfaces"),
    ("MLP-log-arb",  EVAL_DIR / "mlp_log_arb" / "surfaces"),
    ("Conv",         EVAL_DIR / "conv" / "surfaces"),
    ("Conv-log",     EVAL_DIR / "conv_log" / "surfaces"),
    ("Conv-log-arb", EVAL_DIR / "conv_log_arb" / "surfaces"),
])

COLOURS = {
    "MLP": "#1f77b4", "MLP-log": "#9467bd", "MLP-log-arb": "#d62728",
    "Conv": "#2ca02c", "Conv-log": "#8c564b", "Conv-log-arb": "#e377c2",
    "Heston": "#ff7f0e",
}

## 1. Load & Align Surfaces

In [ ]:
# ── Load all VAE surfaces ──
vae_surfaces = {}
grid_spec = None

for name, sdir in VAE_DIRS.items():
    if not sdir.exists():
        print(f"⚠  {name}: not found ({sdir}) — skipping")
        continue
    model = np.load(sdir / "vae_surfaces.npy")
    market = np.load(sdir / "market_surfaces.npy")
    dates = pd.to_datetime(pd.read_csv(sdir / "vae_surface_dates.csv")["date"])
    vae_surfaces[name] = (model, market, dates)
    if grid_spec is None:
        with open(sdir / "grid_spec.json") as f:
            grid_spec = json.load(f)
    print(f"✓ {name:<14} {model.shape}  ({len(dates)} dates)")

# ── Load Heston ──
heston_path = HESTON_DIR / f"{TICKER}_heston_surfaces.npy"
heston_dates_path = HESTON_DIR / f"{TICKER}_heston_surface_dates.csv"
has_heston = heston_path.exists()
if has_heston:
    heston_surf = np.load(heston_path)
    heston_dates = pd.to_datetime(pd.read_csv(heston_dates_path)["date"])
    print(f"✓ {'Heston':<14} {heston_surf.shape}  ({len(heston_dates)} dates)")
else:
    print("⚠ Heston surfaces not found — skipping")

days_grid = np.array(grid_spec["days_grid"])
delta_grid = np.array(grid_spec["delta_grid"])
cp_order = grid_spec["cp_order"]
print(f"\nGrid: {cp_order} × {len(days_grid)} maturities × {len(delta_grid)} deltas")

In [ ]:
# ── Align to common dates ──
def _to_date_set(ds):
    return set(ds.dt.date)

first_name = next(iter(vae_surfaces))
common = _to_date_set(vae_surfaces[first_name][2])
for name, (_, _, dates) in vae_surfaces.items():
    common &= _to_date_set(dates)
if has_heston:
    common &= _to_date_set(heston_dates)
common = sorted(common)
N = len(common)

# Build aligned arrays
model_aligned = {}
for name, (surf, _, dates) in vae_surfaces.items():
    mask = [d in common for d in dates.dt.date]
    model_aligned[name] = surf[mask]

first_vae = next(iter(vae_surfaces.values()))
market_mask = [d in common for d in first_vae[2].dt.date]
market_aligned = first_vae[1][market_mask]

if has_heston:
    heston_mask = [d in common for d in heston_dates.dt.date]
    model_aligned["Heston"] = heston_surf[heston_mask]

aligned_dates = first_vae[2][market_mask].reset_index(drop=True)
NAMES = list(model_aligned.keys())

print(f"Common dates: {N}")
print(f"Range: {common[0]}  →  {common[-1]}")
for name in NAMES:
    print(f"  {name}: {model_aligned[name].shape}")

## 2. Summary Metrics Table

In [ ]:
def compute_metrics(a, b):
    """MSE, MAE, RMSE, max error — NaN-aware."""
    err = a - b
    ae = np.abs(err)
    se = err ** 2
    return {
        "MAE": float(np.nanmean(ae)),
        "MAE (vol pts)": float(np.nanmean(ae) * 100),
        "RMSE": float(np.sqrt(np.nanmean(se))),
        "RMSE (vol pts)": float(np.sqrt(np.nanmean(se)) * 100),
        "MSE": float(np.nanmean(se)),
        "Max": float(np.nanmax(ae)),
        "Valid %": float(np.sum(~np.isnan(err)) / err.size * 100),
    }

rows = []
for name in NAMES:
    m = compute_metrics(model_aligned[name], market_aligned)
    m["Model"] = name
    rows.append(m)

df_summary = pd.DataFrame(rows).set_index("Model")[
    ["MAE", "MAE (vol pts)", "RMSE", "RMSE (vol pts)", "MSE", "Max", "Valid %"]
]

print("=" * 70)
print(f"  {TICKER} — MODEL VS MARKET  (lower is better)")
print("=" * 70)
display(df_summary.style.highlight_min(
    subset=["MAE", "RMSE", "MSE", "Max"],
    axis=0, props="font-weight:bold; background-color:#d4edda"
).format({
    "MAE": "{:.6f}", "RMSE": "{:.6f}", "MSE": "{:.6f}", "Max": "{:.6f}",
    "MAE (vol pts)": "{:.2f}", "RMSE (vol pts)": "{:.2f}", "Valid %": "{:.1f}",
}))

In [ ]:
# Winner
mae_dict = {name: compute_metrics(model_aligned[name], market_aligned)["MAE"]
            for name in NAMES}
winner = min(mae_dict, key=mae_dict.get)
print(f"\n🏆  WINNER (lowest MAE): {winner}  —  MAE = {mae_dict[winner]:.6f}"
      f"  ({mae_dict[winner]*100:.2f} vol pts)")
for n in NAMES:
    if n != winner:
        gap = (mae_dict[n] - mae_dict[winner]) / mae_dict[n] * 100
        print(f"     vs {n}: {gap:.1f}% lower MAE")

## 3. Pairwise Model Differences

In [ ]:
pw_rows = []
for i in range(len(NAMES)):
    for j in range(i + 1, len(NAMES)):
        m = compute_metrics(model_aligned[NAMES[i]], model_aligned[NAMES[j]])
        pw_rows.append({"Pair": f"{NAMES[i]} vs {NAMES[j]}",
                        "MAE": m["MAE"], "RMSE": m["RMSE"]})
df_pw = pd.DataFrame(pw_rows).set_index("Pair")
display(df_pw.style.format({"MAE": "{:.6f}", "RMSE": "{:.6f}"}))

## 4. Error Heatmaps (per model, per option type)

In [ ]:
error_maps = {name: np.nanmean(np.abs(model_aligned[name] - market_aligned), axis=0)
              for name in NAMES}

for c, cp in enumerate(cp_order):
    n_models = len(NAMES)
    fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))
    if n_models == 1:
        axes = [axes]
    vmax = max(error_maps[n][c].max() for n in NAMES
               if not np.all(np.isnan(error_maps[n][c])))

    for i, name in enumerate(NAMES):
        im = axes[i].imshow(error_maps[name][c], aspect="auto", origin="lower",
                            cmap="Reds", vmin=0, vmax=vmax)
        axes[i].set_title(f"{name} ({cp})", fontsize=9)
        axes[i].set_yticks(range(len(days_grid)))
        axes[i].set_yticklabels([int(d) for d in days_grid], fontsize=7)
        if i == 0:
            axes[i].set_ylabel("Maturity (days)")
        axes[i].set_xticks(range(0, len(delta_grid), 3))
        axes[i].set_xticklabels([f"{delta_grid[k]:.2f}" for k in range(0, len(delta_grid), 3)],
                                fontsize=7, rotation=45)
        axes[i].set_xlabel("Delta")
        plt.colorbar(im, ax=axes[i], format="%.3f", shrink=0.85)

    fig.suptitle(f"{TICKER} — Mean Absolute Error — {cp}", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"error_heatmap_{cp}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5. Error Time Series

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for name in NAMES:
    mae_ts = np.nanmean(np.abs(model_aligned[name] - market_aligned), axis=(1, 2, 3))
    ax1.plot(aligned_dates, mae_ts, label=name, alpha=0.6, linewidth=0.8,
             color=COLOURS.get(name))
ax1.set_ylabel("MAE")
ax1.set_title(f"{TICKER} — Daily MAE vs Market")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

window = 20
for name in NAMES:
    mae_ts = np.nanmean(np.abs(model_aligned[name] - market_aligned), axis=(1, 2, 3))
    rolling = pd.Series(mae_ts).rolling(window).mean()
    ax2.plot(aligned_dates, rolling, label=name, linewidth=1.5,
             color=COLOURS.get(name))
ax2.set_ylabel("MAE")
ax2.set_xlabel("Date")
ax2.set_title(f"{window}-Day Rolling Average MAE")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "error_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Per-Maturity and Per-Delta Breakdown

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name in NAMES:
    mae_by_mat = [np.nanmean(np.abs(model_aligned[name][:, :, d, :] - market_aligned[:, :, d, :]))
                  for d in range(len(days_grid))]
    ax1.plot(days_grid, mae_by_mat, 'o-', label=name, color=COLOURS.get(name), markersize=4)
ax1.set_xlabel("Maturity (days)")
ax1.set_ylabel("MAE")
ax1.set_title(f"{TICKER} — MAE by Maturity")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

for name in NAMES:
    mae_by_delta = [np.nanmean(np.abs(model_aligned[name][:, :, :, i] - market_aligned[:, :, :, i]))
                    for i in range(len(delta_grid))]
    ax2.plot(delta_grid, mae_by_delta, 'o-', label=name, color=COLOURS.get(name), markersize=4)
ax2.set_xlabel("Delta")
ax2.set_ylabel("MAE")
ax2.set_title(f"{TICKER} — MAE by Delta")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mae_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Sample Surface Comparisons

In [ ]:
def plot_surface_comparison(idx, cp_idx=0):
    date_str = str(aligned_dates.iloc[idx].date())
    cp = cp_order[cp_idx]
    n_models = len(NAMES)

    fig, axes = plt.subplots(1, n_models + 1, figsize=(4 * (n_models + 1), 4))

    vmin = market_aligned[idx, cp_idx].min()
    vmax = market_aligned[idx, cp_idx].max()
    im0 = axes[0].imshow(market_aligned[idx, cp_idx], aspect="auto", origin="lower",
                         cmap="viridis", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"Market ({cp})", fontsize=9)
    axes[0].set_ylabel("Maturity")
    plt.colorbar(im0, ax=axes[0], format="%.3f", shrink=0.85)

    for i, name in enumerate(NAMES):
        im = axes[i+1].imshow(model_aligned[name][idx, cp_idx], aspect="auto",
                              origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
        mae_val = np.nanmean(np.abs(model_aligned[name][idx, cp_idx] - market_aligned[idx, cp_idx]))
        axes[i+1].set_title(f"{name}\nMAE={mae_val:.4f}", fontsize=9)
        plt.colorbar(im, ax=axes[i+1], format="%.3f", shrink=0.85)

    fig.suptitle(f"{TICKER} — {date_str} — {cp}", fontsize=12)
    plt.tight_layout()
    plt.show()

for idx in [0, N // 2, N - 1]:
    plot_surface_comparison(idx, cp_idx=0)

## 8. Per-Cell Win Count

In [ ]:
for c, cp in enumerate(cp_order):
    print(f"\n{cp}:")
    stacked = np.stack([error_maps[n][c] for n in NAMES], axis=0)
    best_idx = np.nanargmin(stacked, axis=0)

    for i, name in enumerate(NAMES):
        count = np.sum(best_idx == i)
        total = best_idx.size
        print(f"  {name:<14} best in {count:>3}/{total} cells ({100*count/total:.1f}%)")

## 9. Final Summary

In [ ]:
print("=" * 70)
print(f"  {TICKER} — FINAL COMPARISON SUMMARY")
print("=" * 70)
print(f"\n  Test period : {common[0]}  →  {common[-1]}")
print(f"  Surfaces    : {N}")
print(f"  Grid        : {cp_order} × {len(days_grid)} mat × {len(delta_grid)} delta")
print(f"  Models      : {', '.join(NAMES)}")
print()
print(f"  {'Model':<14} {'MAE':>10}  {'(vol pts)':>10}  {'RMSE':>10}  {'(vol pts)':>10}")
print("  " + "-" * 58)

for name in NAMES:
    m = compute_metrics(model_aligned[name], market_aligned)
    print(f"  {name:<14} {m['MAE']:>10.6f}  {m['MAE']*100:>9.2f}%  "
          f"{m['RMSE']:>10.6f}  {m['RMSE']*100:>9.2f}%")

print("  " + "-" * 58)
print(f"\n  🏆  WINNER: {winner}  (MAE = {mae_dict[winner]*100:.2f} vol pts)")
print("=" * 70)

# Save metrics JSON
metrics_out = {name: compute_metrics(model_aligned[name], market_aligned) for name in NAMES}
metrics_out["_meta"] = {
    "ticker": TICKER,
    "n_dates": N,
    "date_range": [str(common[0]), str(common[-1])],
    "winner": winner,
}
with open(OUTPUT_DIR / "comparison_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"\nMetrics saved to {OUTPUT_DIR / 'comparison_metrics.json'}")